# Sound source Localisation
Sound source localisation utilizes the tools already known from the delay and sum beamformer, e.g. [cross correlation](../Basics/CrossCorrelation.ipynb).

The proposed sound source localisation algorithm is explained in the book <cite>Microphone Arrays from M. Brandstein et. al</cite>:

Steering Response Power (SRP) with Phase Transform (PHAT).

## Spatial Setup
In the following, a setup with $4$ microphones and a single sound source at a random position is assumed:

In [1]:
import numpy as np
import wave
import time

### constants and configuration
NumberOfSpatialDimensions = 3
SearchSpaceLimits = [0, 10] # all coordinates in meter
UseAmplitudeInformation = True
UsePHATWeighting = False
SpeedOfSound = 343 # in m/s

# The microphone array is positioned as a tetrahedron
CoordinatesOfMicrophones = []
CoordinatesOfMicrophones.append(np.array([ 1, 1, 1]))
CoordinatesOfMicrophones.append(np.array([ 1,-1,-1]))
CoordinatesOfMicrophones.append(np.array([-1, 1,-1]))
CoordinatesOfMicrophones.append(np.array([-1,-1, 1]))

def GetRandomCoordinatesInSearchSpace():
    result = (SearchSpaceLimits[1] - SearchSpaceLimits[0]) * np.random.rand(NumberOfSpatialDimensions)
    result += SearchSpaceLimits[0]
    return result

# the sound source has a random positioning
CoordinatesOfSoundSource = GetRandomCoordinatesInSearchSpace()

print('Position of the sound source: ', CoordinatesOfSoundSource)

Position of the sound source:  [0.74994319 5.55637158 1.9590343 ]


## Channel and source model
The audio signal $x(n)$ is sent out in all directions evenly. This corresponds to the model of a point source:

The intensity decreases by a square law. By this, it can be followed, that the magnitude decreases linearly by distance as shown in <cite>Acoustic Detection and Tracking of a Class I UAS with a Small Tetrahedral Microphone Array</cite>.

The delay is the distance divided by the speed of sound. The delay in samples is the delay multiplied with the sampling rate.

In [2]:
def ReadWaveAsNumpyArray(Filename):
    """Reads a wave file and returns the audio data as a numpy array along with the sample rate.

    Args:
        Filename (str): Path to the wave file.

    Returns:
        tuple: A tuple containing:
            - numpy.ndarray: The audio data as a numpy array.
            - int: The sample rate of the audio data.
    """
    with wave.open(Filename, 'rb') as wf:
        num_channels = wf.getnchannels()
        sample_width = wf.getsampwidth()
        sample_rate = wf.getframerate()
        num_frames = wf.getnframes()

        raw_data = wf.readframes(num_frames)

        if sample_width == 1:
            dtype = np.uint8  # 8-bit PCM
        elif sample_width == 2:
            dtype = np.int16  # 16-bit PCM
        elif sample_width == 4:
            dtype = np.int32  # 32-bit PCM
        else:
            raise ValueError(f"Unsupported sample width: {sample_width}")

        audio_data = np.frombuffer(raw_data, dtype=dtype) / (2**(8 * sample_width - 1) - 1)

        if num_channels > 1:
            audio_data = audio_data.reshape(-1, num_channels)
            # extract first channel only
            audio_data = audio_data[:, 0]
    return audio_data, sample_rate

x, Fs = ReadWaveAsNumpyArray('../Audio/PferdeSchnaubenNichtDieNase.wav')

def GenerateRecordedSignals(CoordinatesOfSoundSource):
    RecordedSignals = []
    for m in range(len(CoordinatesOfMicrophones)):
        Distance = np.linalg.norm(CoordinatesOfSoundSource - CoordinatesOfMicrophones[m])
        # the sound source is recorded at each microphone with a different delay and amplitude
        if UseAmplitudeInformation:
            Amplitude = 1 / Distance
        else:
            Amplitude = 1.0
        Delay = Distance * Fs / SpeedOfSound # in m/s
        y = Amplitude * np.roll(x, int(Delay)) # roll is a very simple approximation of delayed signals. In real world recordings, the last samples are not inserted at the beginning
        RecordedSignals.append(y)
    return RecordedSignals

RecordedSignals = GenerateRecordedSignals(CoordinatesOfSoundSource)

## Delay estimation
A simple and robust algorithm for delay estimation is the [cross correlation](../Basics/CrossCorrelation.ipynb):

$\varphi_{xy}(m)=\sum_n x(n)\cdot y(n+m)$

In the following it is assumed, that $y(n)$ is a delayed version of $x(n)$

$y(n)=x(n)*\delta(n-\Delta)$

with $\delta(n)$ corresponding to the delta-impulse and $\Delta$ corresponding to the delay in samples. This simplifies the evaluation of the cross correlation to

$\varphi_{xy}(m)=\sum_n x(n)\cdot y(n+m) = \sum_n x(n)\cdot \left(x(n+m)*\delta\left(n-\Delta\right)\right)=\varphi_{xx}(m)*\delta\left(m-\Delta\right)$.

The auto correlation $\varphi_{xx}(m)$ has a single maximum at $m=0$. This maximum corresponds to the energy (or the power) $E_x$ of $x(n)$. Therefore, $\varphi_{xx}(m)*\delta\left(m-\Delta\right)$ has a single maximum at $m=\Delta$, with $\varphi_{xx}(\Delta)=E_x$. Therefore, the delay can be estimated by searching the maximum in the cross correlation.

## Cross correlation in frequency domain
As mentioned in [cross correlation](../Basics/CrossCorrelation.ipynb), the cross correlation can also be evaluated in frequency domain:

$\varphi = \text{DFT}^{-1}\left(\left(\text{DFT}\left(x\right)\right)^* \cdot \text{DFT}\left(y\right)\right)$

## Phase Transform
The Phase Transform (PHAT) makes the source localization more robust against echoes and background noise. It is defined by

$\varphi = \text{DFT}^{-1}\left(\frac{\left(\text{DFT}\left(x\right)\right)^* \cdot \text{DFT}\left(y\right)}{\left|\left(\text{DFT}\left(x\right)\right)^* \cdot \text{DFT}\left(y\right)\right|}\right)$

In [3]:
def EvaluateGeneralizedCrossCorrelation(s, g):
    FFTLen = 2*np.maximum(s.shape[0], g.shape[0])
    Phi = np.fft.fft(g, n=FFTLen) * np.conj(np.fft.fft(s, n=FFTLen))
    if UsePHATWeighting: Phi /= np.abs(Phi) # PHAT weighting
    return np.fft.ifft(Phi).real

def EvaluateDelay(x, y):
    phi = EvaluateGeneralizedCrossCorrelation(x, y)
    FFTLen = phi.shape[0]
    Delay = np.argmax(phi)
    if Delay > FFTLen/2:
        Delay -= FFTLen
    return Delay

# test the delay evaluation  
MaxDelayToTest = x.shape[0]//2
Delay = np.random.randint(MaxDelayToTest) - MaxDelayToTest//2
y = np.roll(x, Delay) # y is delayed by Delay samples
assert EvaluateDelay(x, y) == Delay, 'wrong delay evaluation'

## Evaluation of cross correlations
In a first step, for each pair of microphones $m_1$ and $m_2$ the cross correlation between the two recorded signals is evaluated and stored.

In [4]:
class CGeneralizedCrossCorrelationStorage(object):

    def __init__(self, m1, m2, phi):
        self.__m1 = m1
        self.__m2 = m2
        self.__phi = phi

    def GetM1(self):
        return self.__m1
    
    def GetM2(self):
        return self.__m2
    
    def GetPhi(self):
        return self.__phi

def EvaluateGeneralizedCrossCorrelationForAllMicrophonePairs(RecordedSignals):
    CrossCorrelationValues = []
    for m1 in range(len(RecordedSignals)):
        for m2 in range(m1+1, len(RecordedSignals)):
            CrossCorrelation = EvaluateGeneralizedCrossCorrelation(RecordedSignals[m1], RecordedSignals[m2])
            CrossCorrelationValues.append(CGeneralizedCrossCorrelationStorage(m1, m2, CrossCorrelation))
    return CrossCorrelationValues
        

## Source Localisation by Steering Response Power

The Steering Response Power (SRP) algorithm evaluates for each possible position $P$ of the sound source the corresponding delays of the sound signal to the recorded microphone $m_i$. The corresponding delay is called $\Delta_i$. Then for all pairs of microphones $m_1$ and $m_2$, the two corresponding delays results in a delay between both recorded signals: $\Delta_{m_1,m_2}=\Delta_{m_2}-\Delta_{m_1}$. The cross correlation $\varphi_{m_1,m_2}$ between both recorded signals should have a maximum at position $m=\Delta_{m_1,m_2}$. The value $\varphi_{m_1,m_2}(m=\Delta_{m_1,m_2})$ should correspond to the energy/power of the signal of the sound source. The response at a given position $P$ is the summation of $\varphi_{m_1,m_2}(m=\Delta_{m_1,m_2})$ over all combinations of microphone pairs. The most reasonable source position is found at the maximum response over all possible positions $P$.

The maximization of the response over a grid of possible positions $P$ is called grid search.

In [5]:
def EvalResponseForPoint(Point, CrossCorrelationValues):
    Response = 0
    for CrossCorrelationStorage in CrossCorrelationValues:
        m1 = CrossCorrelationStorage.GetM1()
        m2 = CrossCorrelationStorage.GetM2()
        phi = CrossCorrelationStorage.GetPhi()
        DistanceDifference = np.linalg.norm(Point - CoordinatesOfMicrophones[m2]) - np.linalg.norm(Point - CoordinatesOfMicrophones[m1])
        SampleDifference = int(DistanceDifference / SpeedOfSound * Fs)
        if SampleDifference < 0:
            SampleDifference += phi.shape[0]
        Response += phi[SampleDifference]
    return Response

Delta = 0.1 # in m

def EvalSteeringResponsePower(RecordedSignals):
    MaximumResponse = -np.inf
    CrossCorrelationValues = EvaluateGeneralizedCrossCorrelationForAllMicrophonePairs(RecordedSignals)    
    x = SearchSpaceLimits[0]    
    while x < SearchSpaceLimits[1]:
        y = SearchSpaceLimits[0]
        while y < SearchSpaceLimits[1]:
            z = SearchSpaceLimits[0]
            while z < SearchSpaceLimits[1]:
                Point = np.array([x, y, z])
                Response = EvalResponseForPoint(Point, CrossCorrelationValues)
                if Response > MaximumResponse:
                    MaximumResponse = Response
                    BestPoint = Point
                z += Delta
            y += Delta
        x += Delta
    print('Best point: ', BestPoint, ' with response: ', MaximumResponse)

t1 = time.time()
EvalSteeringResponsePower(RecordedSignals)
t2 = time.time()
print('Time for evaluating the steering response power: ', t2-t1, ' seconds')

CrossCorrelationValues = EvaluateGeneralizedCrossCorrelationForAllMicrophonePairs(RecordedSignals)
Response = EvalResponseForPoint(CoordinatesOfSoundSource, CrossCorrelationValues)
print('Response at true point (', CoordinatesOfSoundSource, '): ', Response)
Point = CoordinatesOfSoundSource / Delta
Point = np.round(Point) * Delta
Response = EvalResponseForPoint(Point, CrossCorrelationValues)
print('Response next to true point (', Point, '): ', Response)

Best point:  [1.  7.  2.5]  with response:  64.4768973409521
Time for evaluating the steering response power:  41.86339855194092  seconds
Response at true point ( [0.74994319 5.55637158 1.9590343 ] ):  63.33471454991951
Response next to true point ( [0.7 5.6 2. ] ):  62.66428120741075


## Source localization by fast search methods
The search over all possible positions by the SRP PHAT algorithm may be too time consuming for real time applications. Faster algorithms can be implemented for searching the maximum response in a range of possible positions $P$. Here, the genetic algorithm is proposed as a faster search algorithm.

A set of parameters (here: the assumed coordinates of the sound source) results in a fitness value (here: the accumulated response over all pairs of microphones). For the genetic algorithm a list of possible sound source coordinates are assumed (forming the so called genome). In each iteration, two possible sound source coordinates are recombined to replace another sound source coordinate. This step is called crossover. Additionally, a single good sound source coordinate is modified by the so called mutation. In order to avoid getting stucked in a local maxima, a random new sound source coordinate is introduced in each iteration, forming the so called random seed. Sound source coordinates with good fitness are chosen as parents, coordinates with less fitness are chosen to be replaced.

By this simple algorithm, the genome moves slowly to a maximum.
The genetic algorithm has the advantage of being relatively robust against getting stuck in local maxima. Additionally, no gradient need to be evaluated.

In [6]:
SizeOfPopulation = 10
MaxIter = 1000

def GetParentIndices(Fitness):
    Limit = np.median(Fitness)
    Parentindex1 = np.random.randint(Fitness.shape[0])
    while Fitness[Parentindex1] < Limit:
        Parentindex1 = np.random.randint(Fitness.shape[0])
    Parentindex2 = np.random.randint(Fitness.shape[0])
    while (Fitness[Parentindex2] < Limit) or (Parentindex2 == Parentindex1):
        Parentindex2 = np.random.randint(Fitness.shape[0])
    return Parentindex1, Parentindex2

def GetKillIndex(Fitness):
    Limit = np.median(Fitness)
    Killindex = np.random.randint(Fitness.shape[0])
    while Fitness[Killindex] > Limit:
        Killindex = np.random.randint(Fitness.shape[0])
    return Killindex
    
def SteeringResponsePowerWithGeneticAlgorithm(RecordedSignals):
    CrossCorrelationValues = EvaluateGeneralizedCrossCorrelationForAllMicrophonePairs(RecordedSignals)
    Genome = np.random.rand(SizeOfPopulation, NumberOfSpatialDimensions)
    Fitness = np.zeros(Genome.shape[0])
    for n in range(Genome.shape[0]):
        Genome[n, :] = GetRandomCoordinatesInSearchSpace()
        Response = EvalResponseForPoint(Genome[n, :], CrossCorrelationValues)
        Fitness[n] = Response

    for n in range(MaxIter):
        # crossover
        Parentindex1, Parentindex2 = GetParentIndices(Fitness)
        Killindex = GetKillIndex(Fitness)
        RandomCoordinates = (Genome[Parentindex1, :] + Genome[Parentindex2, :]) * 0.5
        RandomCoordinates += (Genome[Parentindex1, :] - Genome[Parentindex2, :]) * np.random.randn(Genome.shape[1])
        Genome[Killindex, :] = RandomCoordinates
        Fitness[Killindex] = EvalResponseForPoint(Genome[Killindex, :], CrossCorrelationValues)
        # mutation
        Killindex = GetKillIndex(Fitness)
        RandomCoordinates = GetRandomCoordinatesInSearchSpace()
        Selector = np.random.randn(1, Genome.shape[1]) > 0
        Genome[Killindex, :] = RandomCoordinates * Selector + Genome[Killindex, :] * (1 - Selector)
        Fitness[Killindex] = EvalResponseForPoint(Genome[Killindex, :], CrossCorrelationValues)
        # random seed
        Killindex = GetKillIndex(Fitness)
        RandomCoordinates = GetRandomCoordinatesInSearchSpace()
        Genome[Killindex, :] = RandomCoordinates
        Fitness[Killindex] = EvalResponseForPoint(Genome[Killindex, :], CrossCorrelationValues)

    BestIndex = np.argmax(Fitness)
    return Genome[BestIndex, :], Fitness[BestIndex]

CrossCorrelationValues = EvaluateGeneralizedCrossCorrelationForAllMicrophonePairs(RecordedSignals)
Response = EvalResponseForPoint(CoordinatesOfSoundSource, CrossCorrelationValues)
print('Response at true point (', CoordinatesOfSoundSource, '): ', Response)
t1 = time.time()
EstimatedPosition, EstimatedPower = SteeringResponsePowerWithGeneticAlgorithm(RecordedSignals)
t2 = time.time()
print('Best response: ', EstimatedPower, ' at point (', EstimatedPosition, ')')
print('Time for evaluating the steering response power: ', t2-t1, ' seconds')

Response at true point ( [0.74994319 5.55637158 1.9590343 ] ):  63.33471454991951
Best response:  63.61857067741259  at point ( [1.1306507  7.59888707 2.64193063] )
Time for evaluating the steering response power:  0.48742175102233887  seconds


## Programming exercise
The SRP algorithm is said to be precise regarding the polar $\theta$ and azimuth $\varphi$ angles but not regarding the radius $r$. Implement the conversion from cartesian coordinates into spherical polar coordinates and vice versa, such that this assumption can be verified (or not).

In [7]:
from tqdm import tqdm

def CartesianToPolarCoordinates(x, y, z):
    r = 0
    phi = 0
    theta = 0
    ### solution begins
    r = np.sqrt(x**2 + y**2 + z**2)
    phi = np.arctan2(y, x)
    theta = np.arccos(z / r)
    ### solution ends
    return r, phi, theta

def PolarToCartesianCoordinates(r, phi, theta):
    x = 0
    y = 0
    z = 0
    ### solution begins
    x = r * np.sin(theta) * np.cos(phi)
    y = r * np.sin(theta) * np.sin(phi)
    z = r * np.cos(theta)
    ### solution ends
    return x, y, z

import unittest

class TestProgrammingExercise(unittest.TestCase):

    def test_ConversionCartesianToPolarAndBack(self):
        x0 = np.random.randn(1) * 10
        y0 = np.random.randn(1) * 10
        z0 = np.random.randn(1) * 10
        r, phi, theta = CartesianToPolarCoordinates(x0, y0, z0)
        x1, y1, z1 = PolarToCartesianCoordinates(r, phi, theta)
        self.assertAlmostEqual(x0, x1, delta = 1e-6)
        self.assertAlmostEqual(y0, y1, delta = 1e-6)
        self.assertAlmostEqual(z0, z1, delta = 1e-6)

    def test_ToCartesian1(self):
        r = 2.3
        phi = 0.5
        theta = 1.2
        x1, y1, z1 = PolarToCartesianCoordinates(r, phi, theta)
        self.assertAlmostEqual(1.881, x1, delta = 1e-3)
        self.assertAlmostEqual(1.028, y1, delta = 1e-3)
        self.assertAlmostEqual(0.833, z1, delta = 1e-3)

    def test_ToCartesian2(self):
        r = 0.3
        phi = 0.2
        theta = 1.8
        x1, y1, z1 = PolarToCartesianCoordinates(r, phi, theta)
        self.assertAlmostEqual( 0.286, x1, delta = 1e-3)
        self.assertAlmostEqual( 0.058, y1, delta = 1e-3)
        self.assertAlmostEqual(-0.068, z1, delta = 1e-3)

unittest.main(argv=[''], verbosity=2, exit=False)

test_ConversionCartesianToPolarAndBack (__main__.TestProgrammingExercise.test_ConversionCartesianToPolarAndBack) ... ok
test_ToCartesian1 (__main__.TestProgrammingExercise.test_ToCartesian1) ... ok
test_ToCartesian2 (__main__.TestProgrammingExercise.test_ToCartesian2) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.005s

OK


In [8]:
def MeasureError():
    CoordinatesOfSoundSource = GetRandomCoordinatesInSearchSpace()
    RecordedSignals = GenerateRecordedSignals(CoordinatesOfSoundSource)
    EstimatedPosition, EstimatedPower = SteeringResponsePowerWithGeneticAlgorithm(RecordedSignals)
    r0, phi0, theta0 = CartesianToPolarCoordinates(CoordinatesOfSoundSource[0], CoordinatesOfSoundSource[1], CoordinatesOfSoundSource[2])
    r1, phi1, theta1 = CartesianToPolarCoordinates(EstimatedPosition[0], EstimatedPosition[1], EstimatedPosition[2])
    E_r = r0 - r1
    E_phi = phi0 - phi1
    E_theta = theta0 - theta1
    return E_r, E_phi, E_theta

E_r = np.zeros((100))
E_phi = np.zeros((E_r.shape[0]))
E_theta = np.zeros((E_r.shape[0]))
for n in tqdm(range(E_r.shape[0])):
    E_r[n], E_phi[n], E_theta[n] = MeasureError()
print('Mean error in r: ', np.mean(np.abs(E_r)), ' m')
print('Mean error in phi: ', np.mean(np.abs(E_phi)), ' rad')
print('Mean error in theta: ', np.mean(np.abs(E_theta)), ' rad')
print('standard deviation of error in r: ', np.std(E_r), ' m')
print('standard deviation of error in phi: ', np.std(E_phi), ' rad')
print('standard deviation of error in theta: ', np.std(E_theta), ' rad')

100%|██████████| 100/100 [00:38<00:00,  2.59it/s]

Mean error in r:  2.5100107866802195  m
Mean error in phi:  0.011702325305630477  rad
Mean error in theta:  0.007214288586778344  rad
standard deviation of error in r:  3.215397892599137  m
standard deviation of error in phi:  0.01884960194601778  rad
standard deviation of error in theta:  0.010384722133236454  rad


## Exam preparation

1) Evaluate the finest spatial resolution for $343$ m/s as speed of sound and sampling rate $r=16$ kHz.

## Summary
After working with this Jupyter Notebook you should be able to explain the following topics:

- What is the underlying model of the sound source and the sound propagation, assumed in this jupyter notebook?
- What is the basic idea of the SRP algorithm?
- What is the phase transform?
